# Day 39：Agent 概念与 ReAct 范式（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 6 周**：Agent 基础、工具调用与 LangGraph 状态机  
> **GitLab 工单**：`STARMIND-039` ｜ **开发分支**：`feat/day39-react-agent-from-scratch`  
> **配套 HTML5 课件**：`courseware/day39/index.html` ｜ **独立版本快照**：`versions/day39/`

---

## 🎙️ 老陈开场旁白

早啊各位！欢迎正式跨入第四阶段——Agent 智能体开发。前三十多天我们做的 Prompt 工程和 RAG，本质上都是“人把步骤写死，模型按流水线填空”；从今天开始，我们要把方向盘交出一半给大模型，让它自己看情况决定“下一步该查资料、该算账，还是直接给答案”。今天别急着上任何框架，老陈带大家用最纯粹的 Python 逐行手搓一个 ReAct 引擎。只有自己写过一遍 `Thought -> Action -> Observation` 的主循环，后面看 LangGraph、Dify 源码你才不会发懵。

### 承上启下
- **昨日基线**：昨天（Day 38）我们完成了阶段项目二「企业级 RAG 知识库问答系统」的答辩复盘。大家已经能够熟练搭建“检索 -> 重排 -> 生成”的固定链路（Chain），但在答辩中林产品提了一个尖锐问题：如果员工问“我下周去上海出差 3 天，按公司最新标准最多能报销多少住宿和餐补？”，单纯靠 RAG 只能搜出报销单价，大模型自己心算乘法经常算错，而且无法动态决定先查政策再算账。
- **明日预告**：明天（Day 40）我们会把手写 ReAct 的痛点（正则解析脆弱、工具参数缺强校验）彻底解决，深入 LangChain 的 `@tool` 装饰器、JSON Schema 自动提取、`ToolCallingAgent` 与 `AgentExecutor` 回调观测体系。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day39'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-039）

杭州星辰数科行政与财务部每天收到 300+ 条员工关于差旅标准、补贴核算和年假折算的咨询。纯 RAG 问答只能返回静态条文，员工仍需手工计算天数和金额，体验割裂且易出错。

**今日迭代目标：**
- 梳理 Chain 与 Agent 的架构差异，掌握感知-思考-行动（Observe-Think-Act）闭环原理。
- 零框架纯 Python 实现符合 ReAct 论文规范的推理引擎（Thought / Action / Action Input / Observation / Final Answer）。
- 接入企业内网制度搜索工具与安全四则运算工具，完成复合咨询问题的多步自动求解。

**验收标准：**
- 输入复合差旅核算问题时，Agent 至少完成“先检索制度、再调用计算器、最后给出精确结论”三步闭环。
- 支持 `max_steps` 最大步数熔断机制，遇到未知工具名或格式异常时能向模型反馈纠偏信息而非崩溃。
- 所有核心代码与 `pytest` 单元测试无需外网 API Key 即可在本地 100% 运行通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 什么是真正的 Agent？四个核心要素缺一不可

OpenAI 前应用研究负责人 Lilian Weng 提出过经典的 Agent 公式：`Agent = LLM + Planning（规划） + Memory（记忆） + Tool Use（工具使用）`。在工程落地时，我们用最朴素的标准区分：如果一段代码在运行前你就能确定它一定会调用哪几个函数、调用几次，那就是 Workflow/Chain；如果调用路径依赖大模型在运行时根据中间反馈动态决定，那才是 Agent。

> **⚠️ 老陈避坑**：坑点：很多团队把普通的“意图分类 if-else”包装成 Agent 卖概念，或者反过来把简单固定流程强行交给自由 ReAct 循环，导致原本 1 次 LLM 调用能解决的事变成 5 次调用，延迟翻 5 倍、成本飙升。

### 2. 2. ReAct 协议与 Stop Sequence（停止词）的关键作用

标准 ReAct Prompt 要求模型严格按 `Thought:` -> `Action:` -> `Action Input:` 输出，并在真实调用 OpenAI / DeepSeek / Qwen 接口时传入 `stop=['Observation:', '\nObservation:']`。这样模型一旦吐出 `Action Input` 后准备自己瞎编 `Observation:` 时，API 会立刻截断生成，把控制权交回我们的 Python 代码去执行真实工具。

> **⚠️ 老陈避坑**：坑点：如果忘记配置 `stop` 参数，大模型经常会一口气把 `Observation:` 和 `Final Answer:` 全自己编出来，也就是所谓的“幻觉自问自答”，外部工具根本没被真实调用。

### 3. 3. 为什么严禁在工具里直接用 eval() 做计算器？

网上很多入门教程写计算器工具直接 `return str(eval(expr))`，这在企业内网是 P0 级高危漏洞！只要用户在提问里夹带提示词注入（Prompt Injection），诱导模型传入 `__import__('os').system('cat /etc/passwd')`，你的服务器就直接沦陷。生产级安全计算器必须基于 Python `ast.parse(..., mode='eval')` 白名单遍历，仅允许数字常量与加减乘除幂运算符。

> **⚠️ 老陈避坑**：坑点：还要防指数炸弹，例如 `9**9**9` 会瞬间打满 CPU 和内存，因此 AST 计算器必须对幂运算指数大小做上限校验（如 `abs(right) > 100` 直接拒绝）。


## 💻 实操步骤 1：步骤一：构建企业工具注册表与 AST 安全计算器

- **目标文件**：`versions/day39/src/day39_tools.py`
- **老陈代码旁白**：先别急着写主循环，老陈带你把“武器库”搭稳。这里我们封装两个星辰数科最常用的本地工具：一个是模拟内网规章制度搜索 `internal_search`，另一个是基于 `ast` 抽象语法树白名单的安全计算器 `safe_calculator`，再配一个 `ToolRegistry` 统一生成给大模型看的 Prompt 工具说明书。


In [ ]:
"""Day 39 - 步骤一：企业工具注册表、内网知识搜索与 AST 安全计算器。"""

from __future__ import annotations

import ast
import operator
from dataclasses import dataclass
from typing import Callable, Dict, List


# AST 允许的安全二元运算与一元运算白名单
_BINARY_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}
_UNARY_OPS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}

# 模拟星辰数科企业内网知识库条目
INTERNAL_KB: List[Dict[str, str]] = [
    {
        "title": "杭州星辰数科2026差旅报销标准",
        "keywords": "上海 北京 深圳 一线 差旅 住宿 餐补 报销 标准",
        "content": "一线城市（北京、上海、深圳、广州）住宿标准上限 600 元/晚，餐补 120 元/天，市内交通实报实销上限 80 元/天。",
    },
    {
        "title": "杭州星辰数科二线城市差旅标准",
        "keywords": "杭州 南京 成都 武汉 二线 差旅 住宿 餐补 报销",
        "content": "二线城市（杭州、南京、成都、武汉等）住宿标准上限 450 元/晚，餐补 100 元/天。",
    },
    {
        "title": "星辰数科年假与加班调休管理办法",
        "keywords": "年假 调休 请假 工龄 天数",
        "content": "入职满 1 年享 5 天带薪年假，满 3 年享 10 天；法定节假日加班按日薪 300% 核算。",
    },
]


def _eval_ast_node(node: ast.AST) -> float:
    """递归安全求值 AST 节点，严禁函数调用、属性访问与变量读取。"""
    if isinstance(node, ast.Expression):
        return _eval_ast_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:
        return _UNARY_OPS[type(node.op)](_eval_ast_node(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _BINARY_OPS:
        left = _eval_ast_node(node.left)
        right = _eval_ast_node(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("指数绝对值不能超过 100，已拦截潜在资源耗尽攻击")
        if isinstance(node.op, (ast.Div, ast.FloorDiv, ast.Mod)) and right == 0:
            raise ValueError("除数不能为 0")
        return _BINARY_OPS[type(node.op)](left, right)
    raise ValueError(f"不允许的表达式节点: {type(node).__name__}")


def safe_calculator(expression: str) -> str:
    """安全数学计算器：仅支持数字和 + - * / // % ** 括号运算。"""
    cleaned = expression.strip()
    if not cleaned:
        return "错误：表达式不能为空"
    try:
        tree = ast.parse(cleaned, mode="eval")
        value = _eval_ast_node(tree)
        # 整数结果去除多余的 .0，保留清晰可读性
        if value.is_integer():
            return str(int(value))
        return f"{value:.2f}"
    except Exception as exc:
        return f"计算失败：{exc}"


def internal_search(query: str) -> str:
    """检索星辰数科企业内网规章制度，按关键词命中数排序返回最相关条文。"""
    tokens = [t.strip() for t in query.replace("，", " ").replace(",", " ").split() if t.strip()]
    if not tokens:
        return "未提供检索关键词。"
    scored: List[tuple[int, Dict[str, str]]] = []
    for doc in INTERNAL_KB:
        haystack = f"{doc['title']} {doc['keywords']} {doc['content']}"
        score = sum(1 for tok in tokens if tok in haystack)
        if score > 0:
            scored.append((score, doc))
    if not scored:
        return f"未在内网知识库中找到与“{query}”相关的制度条文。"
    scored.sort(key=lambda item: item[0], reverse=True)
    top_doc = scored[0][1]
    return f"《{top_doc['title']}》：{top_doc['content']}"


@dataclass
class SimpleTool:
    """极简工具元数据包装类。"""

    name: str
    description: str
    func: Callable[[str], str]


class ToolRegistry:
    """工具注册中心：统一管理工具注册、Prompt 文档渲染与安全调用。"""

    def __init__(self) -> None:
        self._tools: Dict[str, SimpleTool] = {}

    def register(self, name: str, description: str, func: Callable[[str], str]) -> None:
        self._tools[name] = SimpleTool(name=name, description=description, func=func)

    def get(self, name: str) -> SimpleTool | None:
        return self._tools.get(name)

    def list_names(self) -> List[str]:
        return list(self._tools.keys())

    def render_prompt_docs(self) -> str:
        """渲染给大模型阅读的工具说明书。"""
        lines = [f"- {t.name}: {t.description}" for t in self._tools.values()]
        return "\\n".join(lines)

    def execute(self, name: str, action_input: str) -> str:
        tool = self._tools.get(name)
        if tool is None:
            available = ", ".join(self.list_names())
            return f"工具调用失败：未知工具 '{name}'。当前可用工具仅包括：[{available}]。"
        try:
            return tool.func(action_input)
        except Exception as exc:
            return f"工具 '{name}' 执行异常：{exc}"


def build_default_registry() -> ToolRegistry:
    """构建星辰数科默认工具集。"""
    registry = ToolRegistry()
    registry.register(
        name="internal_search",
        description="搜索星辰数科内网制度（差旅标准、年假规定等），输入空格分隔的关键词。",
        func=internal_search,
    )
    registry.register(
        name="safe_calculator",
        description="精确计算数学表达式（支持加减乘除与括号），例如 '(600 + 120) * 3'。",
        func=safe_calculator,
    )
    return registry


if __name__ == "__main__":
    reg = build_default_registry()
    print("=== 已注册工具说明书 ===")
    print(reg.render_prompt_docs())
    print("搜索测试:", reg.execute("internal_search", "上海 差旅 住宿"))
    print("计算测试:", reg.execute("safe_calculator", "(600 + 120) * 3"))
    print("恶意注入拦截:", reg.execute("safe_calculator", "__import__('os').system('ls')"))


### 🔍 步骤 1 代码深度解析

1. `_eval_ast_node` 严格采用白名单模式解析数学表达式，拒绝一切 `ast.Call`（函数调用）和 `ast.Name`（变量读取），从根源上杜绝 `eval` 代码注入。
2. `ToolRegistry.execute` 在找不到工具时不会 `raise KeyError`，而是返回包含可用工具列表的自然语言提示——这是 Agent 工程中非常重要的“让模型自我纠错”技巧。
3. `render_prompt_docs()` 会被直接嵌入 ReAct System Prompt 中，工具的 `description` 写得越清晰，模型选错工具的概率就越低。

**预期运行输出参考：**
```text
=== 已注册工具说明书 ===
- internal_search: 搜索星辰数科内网制度（差旅标准、年假规定等），输入空格分隔的关键词。
- safe_calculator: 精确计算数学表达式（支持加减乘除与括号），例如 '(600 + 120) * 3'。
搜索测试: 《杭州星辰数科2026差旅报销标准》：一线城市（北京、上海、深圳、广州）住宿标准上限 600 元/晚，餐补 120 元/天，市内交通实报实销上限 80 元/天。
计算测试: 2160
恶意注入拦截: 计算失败：不允许的表达式节点: Call
```


## 💻 实操步骤 2：步骤二：零框架手写 ReAct 推理-行动循环引擎

- **目标文件**：`versions/day39/src/day39_react_agent.py`
- **老陈代码旁白**：现在来写今天的核心引擎 `ReActAgent`。为了让代码在没有外网 API Key 的本地也能 100% 跑出真实的 multi-step 轨迹，我们定义了一个 `LLMCallable` 接口，并提供 `DeterministicReActLLM` 模拟真实大模型在每一轮看到 `Observation` 后的推理行为；注释里老陈也标明了生产环境替换成 OpenAI/Qwen 客户端的位置。


In [ ]:
"""Day 39 - 步骤二：零框架纯 Python 手写 ReAct Agent 引擎。"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Callable, List, Optional, Tuple

try:
    from src.day39_tools import ToolRegistry, build_default_registry
except ImportError:
    from day39_tools import ToolRegistry, build_default_registry


REACT_SYSTEM_PROMPT = """你是杭州星辰数科的企业智能助手「星辰智脑」。
请严格使用以下 ReAct 格式逐步推理并调用工具回答问题：

可用工具列表：
{tool_docs}

输出格式规范：
Thought: 你的当前思考过程
Action: 要调用的工具名称（必须是 [{tool_names}] 之一）
Action Input: 传给工具的输入参数
Observation: （由系统自动填入工具执行结果，严禁你自行编造）
...（可重复多轮 Thought/Action/Action Input/Observation）
Thought: 我已经掌握足够信息，可以给出最终答复
Final Answer: 给用户的清晰、准确的最终答复

用户问题：{question}
{scratchpad}"""


@dataclass
class AgentStep:
    """记录 ReAct 单步执行轨迹，便于追踪与调试。"""

    step_index: int
    thought: str
    action: Optional[str] = None
    action_input: Optional[str] = None
    observation: Optional[str] = None


@dataclass
class AgentResult:
    """Agent 执行总结果。"""

    final_answer: str
    steps: List[AgentStep] = field(default_factory=list)
    stopped_reason: str = "completed"  # completed | max_steps_reached


def parse_react_output(text: str) -> Tuple[str, Optional[str], Optional[str], Optional[str]]:
    """
    解析模型输出的 ReAct 文本。
    返回: (thought, action, action_input, final_answer)
    """
    thought_match = re.search(r"Thought:\s*(.*?)(?=\n(?:Action:|Final Answer:)|$)", text, re.S)
    thought = thought_match.group(1).strip() if thought_match else ""

    final_match = re.search(r"Final Answer:\s*(.*)", text, re.S)
    if final_match:
        return thought, None, None, final_match.group(1).strip()

    action_match = re.search(r"Action:\s*([^\n]+)", text)
    input_match = re.search(r"Action Input:\s*([^\n]+)", text)
    action = action_match.group(1).strip() if action_match else None
    action_input = input_match.group(1).strip() if input_match else None
    return thought, action, action_input, None


class DeterministicReActLLM:
    """
    本地确定性 ReAct LLM 模拟器（零外网依赖，用于教学演示与 CI 单元测试）。
    生产环境替换为：
        client.chat.completions.create(
            model="qwen-plus", messages=[{"role": "user", "content": prompt}],
            temperature=0, stop=["Observation:", "\\nObservation:"]
        ).choices[0].message.content
    """

    def __call__(self, prompt: str) -> str:
        # 场景 1：上海差旅 3 天报销核算
        if "上海" in prompt and "3" in prompt and "报销" in prompt:
            if "600 元/晚" not in prompt:
                return (
                    "Thought: 员工询问去上海出差 3 天的住宿和餐补报销总额，我需要先查询公司上海差旅标准。\n"
                    "Action: internal_search\n"
                    "Action Input: 上海 差旅 住宿 餐补"
                )
            if "2160" not in prompt:
                return (
                    "Thought: 已查到上海住宿上限 600 元/晚，餐补 120 元/天。为避免心算出错，调用安全计算器算 3 天总额。\n"
                    "Action: safe_calculator\n"
                    "Action Input: (600 + 120) * 3"
                )
            return (
                "Thought: 计算器返回 2160，信息已完整，可以给员工答复。\n"
                "Final Answer: 根据《杭州星辰数科2026差旅报销标准》，上海（一线城市）住宿上限 600 元/晚、餐补 120 元/天，"
                "出差 3 天住宿加餐补最高可报销 2160 元（市内交通另按实报实销上限 80 元/天计算）。"
            )

        # 场景 2：纯数学计算问题
        if "计算" in prompt and "Observation:" not in prompt:
            expr_match = re.search(r"计算[：:\s]*([0-9+\-*/().\s]+)", prompt)
            expr = expr_match.group(1).strip() if expr_match else "1+1"
            return (
                f"Thought: 这是一个纯数学计算请求，直接调用 safe_calculator 工具计算 {expr}。\n"
                f"Action: safe_calculator\n"
                f"Action Input: {expr}"
            )
        if "计算" in prompt and "Observation:" in prompt:
            obs_vals = re.findall(r"Observation:\s*(.+)", prompt)
            ans = obs_vals[-1].strip() if obs_vals else "未知"
            return (
                "Thought: 计算器已经返回精确结果。\n"
                f"Final Answer: 计算结果为 {ans}。"
            )

        return (
            "Thought: 当前问题无需调用外部工具，直接作答。\n"
            "Final Answer: 抱歉，我暂未在知识库中找到对应场景，请联系星辰数科行政部。"
        )


class ReActAgent:
    """零框架手写 ReAct 智能体主引擎。"""

    def __init__(
        self,
        llm: Callable[[str], str],
        registry: ToolRegistry,
        max_steps: int = 5,
    ) -> None:
        self.llm = llm
        self.registry = registry
        self.max_steps = max_steps

    def run(self, question: str) -> AgentResult:
        scratchpad_parts: List[str] = []
        steps: List[AgentStep] = []

        for step_idx in range(1, self.max_steps + 1):
            prompt = REACT_SYSTEM_PROMPT.format(
                tool_docs=self.registry.render_prompt_docs(),
                tool_names=", ".join(self.registry.list_names()),
                question=question,
                scratchpad="\n".join(scratchpad_parts),
            )
            raw_output = self.llm(prompt)
            thought, action, action_input, final_answer = parse_react_output(raw_output)

            if final_answer is not None:
                steps.append(AgentStep(step_index=step_idx, thought=thought))
                return AgentResult(final_answer=final_answer, steps=steps, stopped_reason="completed")

            if not action:
                # 协议解析失败，不崩溃，将纠偏提示写回 Observation 引导下一轮修正
                obs = "格式错误：请严格输出 Action 与 Action Input，或直接输出 Final Answer。"
                steps.append(AgentStep(step_index=step_idx, thought=thought, observation=obs))
                scratchpad_parts.append(f"Thought: {thought}\nObservation: {obs}")
                continue

            observation = self.registry.execute(action, action_input or "")
            steps.append(
                AgentStep(
                    step_index=step_idx,
                    thought=thought,
                    action=action,
                    action_input=action_input,
                    observation=observation,
                )
            )
            scratchpad_parts.append(
                f"Thought: {thought}\nAction: {action}\nAction Input: {action_input}\nObservation: {observation}"
            )

        return AgentResult(
            final_answer=f"已达到最大推理步数上限（{self.max_steps}步），任务安全中止，请缩小问题范围后重试。",
            steps=steps,
            stopped_reason="max_steps_reached",
        )


if __name__ == "__main__":
    agent = ReActAgent(llm=DeterministicReActLLM(), registry=build_default_registry(), max_steps=5)
    result = agent.run("我下周去上海出差 3 天，按公司标准住宿加餐补最多能报销多少钱？")
    for s in result.steps:
        print(f"[Step {s.step_index}] Thought: {s.thought}")
        if s.action:
            print(f"         Action: {s.action}({s.action_input}) -> Obs: {s.observation}")
    print("\n>>> Final Answer:", result.final_answer)


### 🔍 步骤 2 代码深度解析

1. `scratchpad_parts` 充当了 Agent 的短期工作记忆（Scratchpad），每一轮都把 `Thought/Action/Action Input/Observation` 追加进去，重新喂给 LLM。
2. `parse_react_output` 兼容多行 `Thought` 与 `Final Answer`，且当模型漏掉 `Action` 时走“软纠偏”路径，把报错作为 `Observation` 反馈给模型。
3. `max_steps=5` 是生产环境底线：永远不要写 `while True:` 的 Agent，否则模型陷入死循环一晚上能烧掉几千块 Token 费。

**预期运行输出参考：**
```text
[Step 1] Thought: 员工询问去上海出差 3 天的住宿和餐补报销总额，我需要先查询公司上海差旅标准。
         Action: internal_search(上海 差旅 住宿 餐补) -> Obs: 《杭州星辰数科2026差旅报销标准》：一线城市（北京、上海、深圳、广州）住宿标准上限 600 元/晚，餐补 120 元/天，市内交通实报实销上限 80 元/天。
[Step 2] Thought: 已查到上海住宿上限 600 元/晚，餐补 120 元/天。为避免心算出错，调用安全计算器算 3 天总额。
         Action: safe_calculator((600 + 120) * 3) -> Obs: 2160
[Step 3] Thought: 计算器返回 2160，信息已完整，可以给员工答复。

>>> Final Answer: 根据《杭州星辰数科2026差旅报销标准》，上海（一线城市）住宿上限 600 元/晚、餐补 120 元/天，出差 3 天住宿加餐补最高可报销 2160 元（市内交通另按实报实销上限 80 元/天计算）。
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day39/tests/test_day39.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 39 自动化单元测试：验证 AST 安全计算器、工具注册表与 ReAct 引擎全链路。"""

import pytest
from src.day39_tools import build_default_registry, safe_calculator, internal_search
from src.day39_react_agent import ReActAgent, DeterministicReActLLM, parse_react_output


def test_safe_calculator_normal_and_security():
    assert safe_calculator("(600 + 120) * 3") == "2160"
    assert safe_calculator("10 / 4") == "2.50"
    assert "计算失败" in safe_calculator("__import__('os').system('echo pwned')")
    assert "指数绝对值不能超过" in safe_calculator("9 ** 999")
    assert "除数不能为 0" in safe_calculator("100 / 0")


def test_internal_search_hit_and_miss():
    res = internal_search("上海 差旅")
    assert "600" in res and "120" in res
    miss = internal_search("外星人福利")
    assert "未在内网知识库中找到" in miss


def test_react_agent_multi_step_travel_reimbursement():
    agent = ReActAgent(llm=DeterministicReActLLM(), registry=build_default_registry(), max_steps=5)
    result = agent.run("我下周去上海出差 3 天，住宿和餐补最多报销多少？")
    assert result.stopped_reason == "completed"
    assert "2160" in result.final_answer
    assert len(result.steps) == 3
    assert result.steps[0].action == "internal_search"
    assert result.steps[1].action == "safe_calculator"


def test_react_agent_max_steps_circuit_breaker():
    # 模拟一个陷入死循环的模型
    def looping_llm(_prompt: str) -> str:
        return "Thought: 继续查\nAction: internal_search\nAction Input: 上海"

    agent = ReActAgent(llm=looping_llm, registry=build_default_registry(), max_steps=3)
    result = agent.run("测试死循环")
    assert result.stopped_reason == "max_steps_reached"
    assert len(result.steps) == 3


def test_react_agent_unknown_tool_recovery():
    calls = {"n": 0}

    def recovering_llm(prompt: str) -> str:
        calls["n"] += 1
        if calls["n"] == 1:
            return "Thought: 试试不存在的工具\nAction: magic_tool\nAction Input: test"
        assert "未知工具 'magic_tool'" in prompt
        return "Thought: 工具名错了，改用直接回复\nFinal Answer: 已修正错误并安全返回"

    agent = ReActAgent(llm=recovering_llm, registry=build_default_registry(), max_steps=4)
    result = agent.run("测试未知工具恢复")
    assert result.stopped_reason == "completed"
    assert "已修正" in result.final_answer

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：扩展企业日期工作日计算工具

**题目要求：**
为星辰数科 ReAct 工具注册表新增一个 `workday_counter` 工具，接收形如 `'2026-05-01,2026-05-10'` 的起止日期字符串，统计区间内周一至周五的工作日天数并注册到 `ToolRegistry` 中。

**思路提示：** 使用 Python 标准库 `datetime.date.fromisoformat` 与 `timedelta(days=1)` 遍历日期，`weekday() < 5` 即为工作日。


In [ ]:
"""Day 39 作业 1 参考答案：工作日计算工具。"""

from datetime import date, timedelta


def workday_counter(date_range: str) -> str:
    """计算起止日期（含两端）之间的周一至周五工作日天数。"""
    try:
        start_str, end_str = [p.strip() for p in date_range.split(",")]
        start = date.fromisoformat(start_str)
        end = date.fromisoformat(end_str)
        if end < start:
            return "错误：结束日期不能早于开始日期"
        days = 0
        cur = start
        while cur <= end:
            if cur.weekday() < 5:
                days += 1
            cur += timedelta(days=1)
        return f"{start_str} 至 {end_str} 共有 {days} 个工作日"
    except Exception as exc:
        return f"日期格式错误（期望 YYYY-MM-DD,YYYY-MM-DD）：{exc}"


if __name__ == "__main__":
    print(workday_counter("2026-05-01,2026-05-10"))
    assert "6 个工作日" in workday_counter("2026-05-01,2026-05-10")


**答案解析与评分标准：**

核心考察工具的输入边界校验（起止日期倒挂、非法格式异常捕获）与清晰的自然语言返回，确保 LLM 读到 `Observation` 后能直接引用结论。


### 作业 2（企业实战）：为 ReAct 引擎增加单步耗时监控与轨迹 JSON 导出

**题目要求：**
扩展 `ReActAgent`，为每个 `AgentStep` 增加 `latency_ms: float` 字段，统计每轮模型思考与工具执行耗时，并提供 `to_trace_json()` 方法将完整轨迹导出为 JSON 字符串。

**思路提示：** 使用 `time.perf_counter()` 计算毫秒差值，利用 `dataclasses.asdict` 与 `json.dumps(..., ensure_ascii=False, indent=2)` 序列化。


In [ ]:
"""Day 39 作业 2 参考答案：带耗时统计的轨迹追踪器。"""

import json
import time
from dataclasses import asdict, dataclass, field
from typing import Callable, List


@dataclass
class TimedStep:
    step: int
    action: str
    observation: str
    latency_ms: float


@dataclass
class TimedTrace:
    question: str
    steps: List[TimedStep] = field(default_factory=list)

    def to_trace_json(self) -> str:
        return json.dumps(asdict(self), ensure_ascii=False, indent=2)


def run_timed_tool(func: Callable[[str], str], arg: str) -> tuple[str, float]:
    t0 = time.perf_counter()
    res = func(arg)
    elapsed_ms = round((time.perf_counter() - t0) * 1000, 3)
    return res, elapsed_ms


if __name__ == "__main__":
    obs, ms = run_timed_tool(lambda x: f"echo:{x}", "星辰数科")
    trace = TimedTrace(question="测试耗时", steps=[TimedStep(1, "echo", obs, ms)])
    payload = trace.to_trace_json()
    print(payload)
    assert "latency_ms" in payload


**答案解析与评分标准：**

生产环境中 Agent 往往比普通问答慢 3-5 倍，只有把每一步 LLM 与工具的耗时精确到毫秒记录下来，才能定位瓶颈是出在模型慢还是数据库查询慢。


### 作业 3（架构进阶）：实现“连续重复动作”提前熔断器

**题目要求：**
大模型有时会在 `max_steps` 还没耗尽时就连续两次调用完全相同的 `(action, action_input)`。请编写一个 `LoopGuard` 类，当同一动作和参数连续出现 2 次时触发警告并强制拦截。

**思路提示：** 维护一个历史调用签名列表 `(action, normalized_input)`，每次调用前比对最近一次签名。


In [ ]:
"""Day 39 作业 3 参考答案：ReAct 重复动作熔断器。"""

from typing import List, Tuple


class LoopGuard:
    """检测并拦截 Agent 连续重复调用同一工具与参数的死循环。"""

    def __init__(self, max_consecutive_repeats: int = 2) -> None:
        self.max_consecutive_repeats = max_consecutive_repeats
        self._history: List[Tuple[str, str]] = []

    def check_and_record(self, action: str, action_input: str) -> tuple[bool, str]:
        sig = (action.strip(), action_input.strip())
        self._history.append(sig)
        if len(self._history) >= self.max_consecutive_repeats:
            recent = self._history[-self.max_consecutive_repeats :]
            if all(item == sig for item in recent):
                return (
                    False,
                    f"熔断拦截：检测到连续 {self.max_consecutive_repeats} 次重复调用 "
                    f"{sig[0]}({sig[1]})，请更换策略或直接总结已有信息输出 Final Answer。",
                )
        return True, "OK"


if __name__ == "__main__":
    guard = LoopGuard(max_consecutive_repeats=2)
    assert guard.check_and_record("internal_search", "上海")[0] is True
    ok, msg = guard.check_and_record("internal_search", "上海")
    print("第二次相同调用:", ok, msg)
    assert ok is False and "熔断拦截" in msg


**答案解析与评分标准：**

相比傻等 `max_steps=10` 烧完 Token，`LoopGuard` 在第 2 次原地打转时就能立刻给出纠偏信号，直接节省 60% 以上的无效推理成本。


---

## 🎯 老陈收尾复盘

今天干得漂亮！大家亲眼看到了，哪怕不用一行第三方 Agent 框架，200 行标准库 Python 就能跑通“查制度 -> 算总额 -> 汇总答复”的自主推理闭环。记住老陈一句话：Agent 不是魔法，它就是 `while` 循环 + 结构化工具调用 + 严格的熔断边界。下班前把 MR 提上来，明天我们升级到工业级工具 Schema 体系。
